# 🌋 Volcano Compressibility Agent — Colab

[Step-by-step user guide](https://github.com/FreyaMare/volcano-compressibility-agent/blob/main/docs/USER_GUIDE.md)

Type the name of a volcano. The agent

1. **searches the published literature** (Claude + web search) for every input of the seven-step chain — melt compositions, temperatures, H₂O/CO₂/S budgets, storage depths, published deformation sources, elastic moduli — and records the source and status of each number (M measured · U detection limit · A adopted · D generic default);
2. **runs the chain** of the La Fossa thesis — EVo volatile equilibria → magma and chamber compressibility → r<sub>V</sub> → Mogi / Yang / Fialko uplift — plus the inverse analysis, verification checks and sensitivity analyses;
3. **writes a thesis-style PDF report** (abstract, 7 chapters, tables, figures, references, provenance appendix).

You need an Anthropic API key. In Colab, store it once as a secret: 🔑 (left bar) → *Add new secret* → name `ANTHROPIC_API_KEY`.

*Runtime ▸ Run all*, or run the cells one by one to review the data before the analysis.

In [ ]:
#@title 1) Install the agent, EVo and dependencies  { display-mode: "form" }
#@markdown The code is downloaded from GitHub. If that fails (no internet access to GitHub, or the
#@markdown repository is private), a button appears to upload **volcano-compressibility-agent.zip** instead.
REPO_URL = "https://github.com/FreyaMare/volcano-compressibility-agent"  #@param {type:"string"}
import os, sys, subprocess, glob, zipfile
if not os.path.isdir("volcano-agent"):
    ok = False
    if REPO_URL.strip():
        ok = subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL.strip(), "volcano-agent"]).returncode == 0
    if not ok:
        zips = glob.glob("volcano-compressibility-agent*.zip")
        if not zips:
            from google.colab import files
            print("Choose the file volcano-compressibility-agent.zip ...")
            zips = list(files.upload().keys())
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall("_unzip")
        inner = [d for d in glob.glob("_unzip/*") if os.path.isdir(d)]
        os.rename(inner[0] if inner else "_unzip", "volcano-agent")
print("Installing packages (1-2 minutes)...")
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", "volcano-agent/requirements.txt"], check=True)
sys.path.insert(0, os.path.abspath("volcano-agent"))
from volcano_agent import physics
print("Ready. EVo installed at", physics.ensure_evo())

In [ ]:
#@title 2) Volcano and settings  { display-mode: "form" }
VOLCANO = "Campi Flegrei"  #@param {type:"string"}
MODEL = "claude-opus-5-5"  #@param ["claude-opus-5-5", "claude-sonnet-5-5"]
RESEARCH_DEPTH = "standard"  #@param ["quick", "standard", "thorough"]
TARGET_UPLIFT_MM = 10.0  #@param {type:"number"}
SCREEN_MPA = 10.0  #@param {type:"number"}

API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        from getpass import getpass
        API_KEY = getpass("Anthropic API key: ")
print("Volcano:", VOLCANO, "| model:", MODEL, "| key:", "set" if API_KEY else "MISSING")

In [ ]:
#@title 3) Literature research (a few minutes)  { display-mode: "form" }
import pandas as pd
from IPython.display import display
from volcano_agent.pipeline import research
ds, usage = research(VOLCANO, api_key=API_KEY, model=MODEL, depth=RESEARCH_DEPTH, on_event=print)
print("\n", usage.summary())

def show_dataset(ds):
    rows = []
    for m in ds.magmas:
        rows.append(dict(item=f"{m.key} composition", value=f"SiO2 {m.oxides.get('SIO2', 'n/a')}", status=m.composition_status, source=m.composition_source))
        for n, q in (("T °C", m.T_C), ("H2O wt%", m.H2O_wt), ("CO2 wt%", m.CO2_wt), ("S wt%", m.S_wt)):
            rows.append(dict(item=f"{m.key} {n}", value=q.value, status=q.status, source=q.source))
    for L in ds.levels:
        rows.append(dict(item=f"{L.name}: depth km ({L.resident}/{L.injected})", value=L.depth_km.value, status=L.depth_km.status, source=L.depth_km.source))
    s = ds.deformation_source
    if s is not None:
        rows.append(dict(item=f"Published source {s.model}: depth km", value=s.depth_km.value, status=s.depth_km.status, source=s.source))
        rows.append(dict(item="Published source: dV m3", value=s.dV_m3.value, status=s.dV_m3.status, source=s.dV_m3.source))
    display(pd.DataFrame(rows))
    for g in ds.data_gaps:
        print("•", g)
show_dataset(ds)

In [ ]:
#@title 4) (Optional) review or correct the data  { display-mode: "form" }
#@markdown Edit values in code before running the analysis, for example:
#@markdown `ds.magma("RHYOLITE").H2O_wt.value = 1.3` · `ds.levels[0].depth_km.value = 2.5` · `ds.deformation_source = None`.
#@markdown Any value you set by hand should also get a status and a source:
#@markdown `ds.magma("RHYOLITE").H2O_wt.status = "A"; ds.magma("RHYOLITE").H2O_wt.source = "My et al. (2024)"`.
# --- your edits here ---

with open(f"{ds.slug}_dataset.json", "w") as f:
    f.write(ds.model_dump_json(indent=1))
print("Dataset saved as", f"{ds.slug}_dataset.json")

In [ ]:
#@title 5) Run the seven-step chain and write the PDF report  { display-mode: "form" }
from IPython.display import Image as IImage
from volcano_agent.pipeline import analyse_and_report
from volcano_agent import figures as F
rr = analyse_and_report(ds, api_key=API_KEY, model=MODEL, target_mm=TARGET_UPLIFT_MM,
                        screen_MPa=SCREEN_MPA, usage=usage, on_event=print)
R = rr.results
display(R["res"][["label", "depth_km", "P_MPa", "phi", "rV", "rV_eq", "uz1_0km_mm", "uz3_0km_mm", "uz3_2km_mm"]])
display(R["inverse"][["label", "branch", "rV", "Ve", "dVc", "dP_MPa", "within_screen"]])
display(IImage(F.profiles(rr.analysis, R, R["thin_crack"])))
pdf_name = f"{rr.dataset.slug}_report.pdf"
with open(pdf_name, "wb") as f:
    f.write(rr.pdf)
print("Report written:", pdf_name)
try:
    from google.colab import files
    files.download(pdf_name)
except Exception:
    pass

---
### Demo without an API key
The cell below runs the chain on the **La Fossa thesis dataset** (Scenario A) and builds the report with placeholder text instead of the model-written chapters. It reproduces the thesis numbers exactly.

In [ ]:
#@title Demo: La Fossa (no API key needed)  { display-mode: "form" }
from volcano_agent.pipeline import analyse_and_report, load_reference
demo = analyse_and_report(load_reference("lafossa"), write=False, on_event=print)
display(demo.results["res"][["label", "rV", "uz3_0km_mm"]])
with open("La_Fossa_demo_report.pdf", "wb") as f:
    f.write(demo.pdf)
print("La_Fossa_demo_report.pdf written")